# Benchmark self-hosted: gemma3n vs gemma2:2b — Expensum

Notebook autosufficiente (nessuna dipendenza dal repo privato) per testare i modelli
self-hosted su GPU gratuita Colab, sui 4 task reali del bot: **parsing testo, report
settimanale/mensile, parsing foto (vision), parsing audio**.

**Prerequisiti prima di eseguire:**
1. `Runtime -> Change runtime type -> T4 GPU`
2. Accetta la licenza Gemma su Hugging Face per **entrambi** i modelli (una volta sola,
   serve un account HF): https://huggingface.co/google/gemma-3n-e2b-it e
   https://huggingface.co/google/gemma-2-2b-it
3. Esegui le celle in ordine. In fondo trovi il download del file
   `gemma_colab_results.json` — dallo a Claude per lo scoring finale (fatto in locale
   con le stesse funzioni usate per Mistral, cosi' il confronto resta omogeneo).

Ogni cella salva i risultati **grezzi** (testo generato dal modello, non ancora
punteggiato) — lo scoring/confronto con Mistral avviene fuori da qui.

In [ ]:
!pip install -q -U "transformers>=4.53.0" accelerate gTTS pillow bitsandbytes

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # serve un token HF con accesso accettato alla licenza Gemma

In [ ]:
import torch, json, time, gc, os
from datetime import date

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU disponibile:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

RESULTS = {"text_parsing": {}, "reports": {}, "photo": {}, "audio": {}}

def free_model(*names: str):
    """Libera VRAM per davvero: cancella i nomi dal namespace GLOBALE del notebook,
    non variabili locali (del o su un parametro locale NON tocca l'oggetto originale
    referenziato altrove — la VRAM resterebbe occupata e il modello successivo
    verrebbe silenziosamente spezzato su CPU da device_map="auto")."""
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM allocata dopo pulizia: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Costanti — copiate 1:1 dal codice di produzione

`SYSTEM_PROMPT` e `CATEGORIES` vengono da `src/expense_parser.py` e
`src/workspace_bootstrapper.py`. `WEEKLY_PROMPT_DATA`/`MONTHLY_PROMPT_DATA` sono
l'output GIA' CALCOLATO da `WeeklyPlanBuilder`/`MonthlyPlanBuilder` sugli stessi dati
mock usati per il test locale Mistral — così il confronto è sullo stesso identico
input, non ricalcolato qui.

In [ ]:
SYSTEM_PROMPT = 'Sei un assistente finanziario. Il tuo compito è estrarre informazioni finanziarie da messaggi scritti in modo informale in italiano. Devi capire dal contesto se si tratta di un\'uscita (spesa), di un\'entrata (stipendio, regalo, rimborso, bonifico ricevuto) o di uno spostamento di denaro tra due conti dello stesso utente (trasferimento/giroconto).\n\nREGOLE TASSATIVE:\n1. Rispondi SOLO con un oggetto JSON valido, senza altro testo.\n2. Ogni transazione è un oggetto con esattamente questi 8 campi:\n   - "type": stringa. Deve essere ASSOLUTAMENTE "expense" (se è una spesa/uscita), "income" (se è un introito/entrata) oppure "transfer" (se l\'utente sposta soldi da un SUO conto a un ALTRO SUO conto — es. "sposta", "trasferisci", "giroconto", "bonifico interno").\n   - "amount": numero (float). DEVI interpretare anche i numeri a parole. Esempi: "150 mila euro" -> 150000.0, "cento euro" -> 100.0, "15" -> 15.0. Se non specificato, metti null.\n   - "category": stringa. RIGOROSAMENTE UNA SOLA voce dalla lista fornita di Categorie. Se `type` è "income" o "transfer", metti null.\n   - "income_type": stringa. SOLO se `type` è "income": scegli OBBLIGATORIAMENTE una voce tra [Stipendio, Bonus, Rimborso, Regalo, Cashback, Dividendi, Affitto, Interessi, Mance, Ripetizioni, Paghetta]. Se `type` è "expense" o "transfer", metti null.\n   - "name": stringa, breve descrizione (es. "Pizza amici", "Stipendio", "Busta Nonna", "Trasferimento"). Max 3 parole.\n   - "date": stringa in formato "YYYY-MM-DD". Ragiona SEMPRE a partire dalla "Data di oggi" fornita nel prompt utente, che è il tuo unico ancoraggio temporale (non conosci la data reale). Interpreta QUALSIASI riferimento temporale relativo, a qualunque distanza: "ieri" (oggi-1), "ieri l\'altro"/"altro ieri"/"avantieri" (oggi-2), "N giorni fa", "N settimane fa", "N mesi fa", nomi di giorni della settimana con "scorso" (es. "lunedì scorso" → il lunedì più recente nel passato, mai oggi stesso), "il mese scorso", date esplicite senza anno (es. "il 12 marzo" → stesso anno della data di oggi, o l\'anno precedente se quel giorno è ancora futuro rispetto a oggi). Se non specificata alcuna data, usa la data di oggi fornita.\n   - "from_account": stringa. SOLO se `type` è "transfer": il nome del conto di ORIGINE così come scritto/menzionato dall\'utente (es. "conto corrente", "Revolut"). Se non menzionato o `type` non è "transfer", metti null. NON inventare un nome se l\'utente non lo scrive.\n   - "to_account": stringa. SOLO se `type` è "transfer": il nome del conto di DESTINAZIONE così come scritto/menzionato dall\'utente. Se non menzionato o `type` non è "transfer", metti null. NON inventare un nome se l\'utente non lo scrive.\n3. Se il messaggio NON sembra una transazione finanziaria, rispondi con: {"error": "not_expense"}\n4. NON inventare importi né nomi di conto.\n5. MULTI-TRANSAZIONE — decidi tu se il messaggio ne descrive una sola o più di una:\n   - Se il messaggio descrive UNA SOLA transazione, rispondi con l\'oggetto singolo come negli esempi sotto (nessun wrapper).\n   - Se il messaggio descrive PIÙ transazioni distinte (es. più importi/voci separati, anche con date diverse tra loro), rispondi con un oggetto wrapper: {"transactions": [ {oggetto transazione 1}, {oggetto transazione 2}, ... ]}, un elemento per ogni transazione rilevata, ciascuno con tutti e 8 i campi sopra. Ogni transazione può avere una propria "date" indipendente (tutte ancorate alla stessa "Data di oggi").\n   - Non usare MAI il wrapper "transactions" per una singola transazione.\n\nESEMPI SPESE (transazione singola):\n- "ieri pizza 25€" → {"type": "expense", "amount": 25.0, "category": "Ristoranti", "income_type": null, "name": "Pizza", "date": "2026-03-20", "from_account": null, "to_account": null}\n- "netflix" → {"type": "expense", "amount": null, "category": "Abbonamenti", "income_type": null, "name": "Netflix", "date": "2026-03-21", "from_account": null, "to_account": null}\n- "ho speso 67 euro al supermercato" → {"type": "expense", "amount": 67.0, "category": "Supermercato", "income_type": null, "name": "Spesa supermercato", "date": "2026-03-21", "from_account": null, "to_account": null}\n- "3 settimane fa ho pagato 80 euro di dentista" (oggi=2026-04-10) → {"type": "expense", "amount": 80.0, "category": "Salute", "income_type": null, "name": "Dentista", "date": "2026-03-20", "from_account": null, "to_account": null}\n\nESEMPI ENTRATE (transazione singola):\n- "ho ricevuto lo stipendio 1500" → {"type": "income", "amount": 1500.0, "category": null, "income_type": "Stipendio", "name": "Stipendio", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "busta paga marzo 1800 euro" → {"type": "income", "amount": 1800.0, "category": null, "income_type": "Stipendio", "name": "Busta paga marzo", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "rimborso spese 120€" → {"type": "income", "amount": 120.0, "category": null, "income_type": "Rimborso", "name": "Rimborso spese", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "mi hanno pagato 500" → {"type": "income", "amount": 500.0, "category": null, "income_type": "Stipendio", "name": "Pagamento ricevuto", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "bonifico ricevuto 300 euro" → {"type": "income", "amount": 300.0, "category": null, "income_type": "Stipendio", "name": "Bonifico ricevuto", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "regalo compleanno 50 euro" → {"type": "income", "amount": 50.0, "category": null, "income_type": "Regalo", "name": "Regalo compleanno", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "cashback paypal 12 euro" → {"type": "income", "amount": 12.0, "category": null, "income_type": "Cashback", "name": "Cashback PayPal", "date": "2026-03-22", "from_account": null, "to_account": null}\n- "dividendi azionari 200 euro" → {"type": "income", "amount": 200.0, "category": null, "income_type": "Dividendi", "name": "Dividendi azionari", "date": "2026-03-22", "from_account": null, "to_account": null}\n\nESEMPI TRASFERIMENTI (transazione singola):\n- "sposta 100 euro dal conto corrente al conto risparmio" → {"type": "transfer", "amount": 100.0, "category": null, "income_type": null, "name": "Trasferimento", "date": "2026-03-22", "from_account": "conto corrente", "to_account": "conto risparmio"}\n- "giroconto di 50€ da Revolut a Fineco" → {"type": "transfer", "amount": 50.0, "category": null, "income_type": null, "name": "Giroconto", "date": "2026-03-22", "from_account": "Revolut", "to_account": "Fineco"}\n- "bonifico interno 300 euro verso il conto risparmio" → {"type": "transfer", "amount": 300.0, "category": null, "income_type": null, "name": "Bonifico interno", "date": "2026-03-22", "from_account": null, "to_account": "conto risparmio"}\n- "trasferisci 200 euro sul conto di risparmio" → {"type": "transfer", "amount": 200.0, "category": null, "income_type": null, "name": "Trasferimento", "date": "2026-03-22", "from_account": null, "to_account": "conto di risparmio"}\n\nESEMPI MULTI-TRANSAZIONE:\n- "ho speso 10 euro per il pranzo, 20 di benzina e 5 di caffè ieri" (oggi=2026-04-10) → {"transactions": [\n  {"type": "expense", "amount": 10.0, "category": "Ristoranti", "income_type": null, "name": "Pranzo", "date": "2026-04-09", "from_account": null, "to_account": null},\n  {"type": "expense", "amount": 20.0, "category": "Trasporti", "income_type": null, "name": "Benzina", "date": "2026-04-09", "from_account": null, "to_account": null},\n  {"type": "expense", "amount": 5.0, "category": "Bar", "income_type": null, "name": "Caffè", "date": "2026-04-09", "from_account": null, "to_account": null}\n]}\n- "oggi 15 euro di spesa e ieri ho ricevuto un rimborso di 50 euro" (oggi=2026-04-10) → {"transactions": [\n  {"type": "expense", "amount": 15.0, "category": "Supermercato", "income_type": null, "name": "Spesa", "date": "2026-04-10", "from_account": null, "to_account": null},\n  {"type": "income", "amount": 50.0, "category": null, "income_type": "Rimborso", "name": "Rimborso", "date": "2026-04-09", "from_account": null, "to_account": null}\n]}\n'
CATEGORIES = ['Affitto', 'Mutuo', 'Bollette luce', 'Bollette gas', 'Acqua', 'Condominio', 'Casa Vario', 'Supermercato', 'Mercato', 'Ristorante', 'Bar', 'Food delivery', 'Carburante', 'Mezzi pubblici', 'Taxi', 'Parcheggio', 'Auto Vario', 'Farmacia', 'Medicina', 'Telefono', 'Internet', 'Abbigliamento', 'Scarpe', 'Parrucchiere', 'Igiene personale', 'Assicurazioni', 'Banca', 'Regali', 'Altro']
TEXT_CASES = [{'msg': 'pizza 15', 'type': 'expense', 'amount': 15.0, 'category': 'Ristorante'}, {'msg': 'benzina 40 euro', 'type': 'expense', 'amount': 40.0, 'category': 'Carburante'}, {'msg': 'affitto', 'type': 'expense', 'amount': None, 'category': 'Affitto'}, {'msg': 'netflix', 'type': 'expense', 'amount': None, 'category': None}, {'msg': 'ieri sera mangiato fuori tipo 40 con i ragazzi', 'type': 'expense', 'amount': 40.0, 'category': 'Ristorante', 'date': '2026-04-09'}, {'msg': 'ho preso caffè e cornetto stamattina 2.70', 'type': 'expense', 'amount': 2.7, 'category': 'Bar'}, {'msg': 'spesa esselunga ieri circa 90', 'type': 'expense', 'amount': 90.0, 'category': 'Supermercato', 'date': '2026-04-09'}, {'msg': 'cento euro di spesa', 'type': 'expense', 'amount': 100.0, 'category': 'Supermercato'}, {'msg': '100 euro', 'type': None}, {'msg': 'arrivato lo stipendio 1800', 'type': 'income', 'amount': 1800.0, 'income_type': 'Stipendio'}, {'msg': 'bonus 300€', 'type': 'income', 'amount': 300.0, 'income_type': 'Bonus'}, {'msg': 'mi hanno rimborsato il viaggio 120', 'type': 'income', 'amount': 120.0, 'income_type': 'Rimborso'}, {'msg': 'regalo di compleanno 50 euro dalla nonna', 'type': 'income', 'amount': 50.0, 'income_type': 'Regalo'}, {'msg': 'sposta 100 euro dal conto corrente al conto risparmio', 'type': 'transfer', 'amount': 100.0, 'from_account': 'conto corrente', 'to_account': 'conto risparmio'}, {'msg': 'giroconto di 50€ da Revolut a Fineco', 'type': 'transfer', 'amount': 50.0, 'from_account': 'Revolut', 'to_account': 'Fineco'}, {'msg': '3 settimane fa ho pagato 80 euro di dentista', 'type': 'expense', 'amount': 80.0, 'category': 'Medicina', 'date': '2026-03-20'}, {'msg': 'il mese scorso bolletta della luce 95 euro', 'type': 'expense', 'amount': 95.0, 'category': 'Bollette luce', 'date': '2026-03-01'}, {'msg': 'ok grazie', 'type': None}, {'msg': "domani devo pagare l'affitto", 'type': 'expense', 'amount': None, 'category': 'Affitto', 'date': '2026-04-11'}, {'msg': 'ho speso 10 euro per il pranzo, 20 di benzina e 5 di caffè ieri', 'type': 'multi', 'count': 3, 'date': '2026-04-09'}, {'msg': 'oggi 15 euro di spesa e ieri ho ricevuto un rimborso di 50 euro', 'type': 'multi', 'count': 2}, {'msg': 'farmacia ieri 23.50', 'type': 'expense', 'amount': 23.5, 'category': 'Farmacia', 'date': '2026-04-09'}, {'msg': 'ricarica telefono 15', 'type': 'expense', 'amount': 15.0, 'category': 'Telefono'}]
PHOTO_CASES = [{'store': 'ESSELUNGA', 'items': [('Pane integrale', '1.29'), ('Latte fresco', '1.45'), ('Uova x6', '2.89')], 'total': '5.63', 'date': '15/04/2026', 'category': 'Supermercato'}, {'store': 'SHELL', 'items': [('Benzina 30L', '51.90'), ('Acqua 0.5L', '1.50')], 'total': '53.40', 'date': '12/04/2026', 'category': 'Carburante'}, {'store': "FARMACIA SANT'ANNA", 'items': [('Aspirina 20cpr', '3.50'), ('Vitamina C', '6.90')], 'total': '10.40', 'date': '10/04/2026', 'category': 'Farmacia'}, {'store': 'TRATTORIA DA LUIGI', 'items': [("Tagliatelle al ragu'", '12.00'), ('Vino della casa', '4.50'), ('Coperto', '2.00')], 'total': '18.50', 'date': '08/04/2026', 'category': 'Ristorante'}, {'store': 'ZARA', 'items': [('T-shirt bianca S', '17.95'), ('Jeans slim 32', '39.95')], 'total': '57.90', 'date': '05/04/2026', 'category': 'Abbigliamento'}, {'store': 'AUTOGRILL', 'items': [('Panino prosciutto', '5.50'), ("Caffe' doppio", '2.10')], 'total': '7.60', 'date': '01/04/2026', 'category': 'Bar'}]
AUDIO_CASES = [{'text': 'ho speso diciotto euro per il pranzo al lavoro oggi', 'type': 'expense', 'amount': 18.0, 'category': 'Ristorante'}, {'text': 'ho ricevuto un rimborso di ventiquattro euro dal collega', 'type': 'income', 'amount': 24.0, 'income_type': 'Rimborso'}, {'text': 'trentacinque euro di benzina ieri', 'type': 'expense', 'amount': 35.0, 'category': 'Carburante'}, {'text': 'arrivato lo stipendio di milleottocento euro', 'type': 'income', 'amount': 1800.0, 'income_type': 'Stipendio'}, {'text': 'dodici euro e cinquanta di farmacia', 'type': 'expense', 'amount': 12.5, 'category': 'Farmacia'}, {'text': 'sposta cento euro dal conto corrente al conto risparmio', 'type': 'transfer', 'amount': 100.0, 'from_account': 'conto corrente', 'to_account': 'conto risparmio'}]
WEEKLY_TEMPLATE = '\nSei un Coach Finanziario Personale specializzato in finanza quotidiana.\nHai ricevuto i seguenti dati sull\'andamento finanziario settimanale e mensile di un cliente:\n\n--- DATI ANALIZZATI ---\n{plan_data}\n\n# ISTRUZIONI DI RAGIONAMENTO (Chain of Thought)\n\nPrima di produrre il piano, analizza internamente:\n1. Quali categorie sono critiche (status=attenzione o sforato: proiezione a fine mese supera il budget)?\n2. Quali categorie mostrano miglioramento: rinforza il comportamento positivo.\n3. Gli abbonamenti in scadenza riducono il budget netto: tienine conto nei consigli.\n4. I pattern giornalieri indicano quando l\'utente spende di più: sfruttali per i consigli.\n5. Se l\'utente ha indicato vincoli espliciti, rispettali e costruisci attorno a loro.\n6. Se la spesa della settimana scorsa per una categoria è €0.00 (indicato nel dato con [spesa=0: non mostrare avvisi negativi]), NON mostrare avvisi negativi per quella categoria — usa ⬜ nella tabella.\n\n# OUTPUT RICHIESTO\n\nProduci ESATTAMENTE il seguente piano settimanale in Markdown per Telegram.\nUsa *grassetto* solo con singolo asterisco.\nNON inventare numeri non presenti nei dati.\nIl linguaggio deve essere in italiano, il tono deve essere amichevole e informale. \nSii spiritoso e simpatico MA NON MENTIRE MAI SUI DATI.\nSii specifico e concreto: USA I DATI FORNITI E REALI, non consigli generici.\nTono prospettico: "questa settimana punta a...", non retrospettivo.\n\n---\n\n📅 *Piano Settimana {week_label}*\n\n🔔 *Abbonamenti in uscita questa settimana:*\n[Se ci sono abbonamenti nei dati: elencali con importo e data di scadenza]\n[Se NON ci sono abbonamenti in scadenza: scrivi "Nessun abbonamento in scadenza questa settimana. 🎉"]\n\n---\n\n🎯 *Obiettivi per categoria*\n_Gli obiettivi sono calcolati sul budget mensile impostato dall\'utente. Può cambiarlo con /modify_ 🎛️\n\nGenera una tabella in code block (```) con ESATTAMENTE 4 colonne e larghezza MASSIMA 34 caratteri per riga (indispensabile per non andare a capo su Telegram mobile).\nColonne: Categoria (13 car, ljust, tronca con "." se più lungo) | Scorsa (7 car, rjust, €XX.XX) | /sett (6 car, rjust, €XXX/s senza decimali) | T (emoji stato).\nLegenda T: usa il campo "status" presente nei dati per ogni categoria.\n  • status=ok → ✅ (proiezione fine mese entro il budget)\n  • status=attenzione → ⚠️ (al ritmo attuale supererai il budget a fine mese)\n  • status=sforato → 🔴 (budget mensile già superato)\n  • spesa settimana scorsa = €0.00 → ⬜ (indipendentemente dallo status)\nDopo il code block aggiungi SEMPRE questa legenda in corsivo:\n_✅ in target · ⚠️ a questo ritmo esci dal budget · ⬜ nessuna spesa sett. scorsa_\nEsempio formato (rispetta ESATTAMENTE questi allineamenti):\n```\nCategoria      Scorsa   /sett T\n───────────────────────────────\nAlimentari     €26.84  €86/s ✅\nRistoranti.    €21.41  €46/s ⚠️\nTrasporti       €0.00  €37/s ⬜\nSvago & Int.  €120.00  €75/s 🔴\n```\n_✅ in target · ⚠️ a questo ritmo esci dal budget · ⬜ nessuna spesa sett. scorsa_\n\n---\n\n💡 *3 Consigli per questa settimana*\n\nBasati ESCLUSIVAMENTE sui pattern e anomalie nei dati:\n\n1️⃣ [Consiglio specifico con giorno/categoria/importo reale dai dati — es. "il sabato spendi il 35% del tuo budget settimanale alimentare: pianifica la spesa in anticipo"]\n2️⃣ [Consiglio specifico]\n3️⃣ [Consiglio specifico]\n\n---\n✨ _Generato con AI — verifica sempre i numeri_\n_Scrivi in modo naturale per modificare un obiettivo_\n'
MONTHLY_TEMPLATE = '\nSei un Coach Finanziario Personale specializzato in pianificazione mensile.\nHai ricevuto i seguenti dati per pianificare il mese di {month_label}:\n\n--- DATI ANALIZZATI ---\n{plan_data}\n\n# ISTRUZIONI DI RAGIONAMENTO (Chain of Thought)\n\nPrima di produrre il piano, analizza internamente:\n1. Quali categorie mostrano trend in crescita negli ultimi 3 mesi? Richiedono attenzione o un piano di contenimento?\n2. Quali categorie sono in calo? Rinforza questo comportamento positivo nel piano.\n3. L\'obiettivo di risparmio è realistico rispetto alle entrate previste e alle spese storiche?\n4. Gli abbonamenti in scadenza riducono il budget disponibile: quantificali nei consigli.\n5. Se l\'utente ha indicato vincoli espliciti, rispettali e costruisci il piano attorno a loro.\n\n# OUTPUT RICHIESTO\n\nProduci ESATTAMENTE il seguente piano mensile in Markdown per Telegram.\nUsa *grassetto* solo con singolo asterisco.\nNON inventare numeri non presenti nei dati.\nSii specifico e concreto: usa i dati reali, non consigli generici.\nTono prospettico: "questo mese punta a...", non retrospettivo.\n\n---\n\n📅 *Piano Mensile {month_label}*\n\n💰 Previsione entrate: *€[income_forecast]*\n🎯 Obiettivo risparmio: *€[savings_goal]*\n🏦 Saldo conti attuale: *€[total_accounts]*\n\n[Se ci sono abbonamenti in scadenza:]\n🔔 *Abbonamenti in uscita questo mese:*\n[lista abbonamenti con importo e data di scadenza]\n\n---\n\n🎯 *Budget per categoria*\n\n[Per ogni categoria con target, una riga: nome, target €/mese, e una nota breve basata sul trend. Es:]\n• *Alimentari* — €X/mese _(stabile negli ultimi 6 mesi)_\n• *Ristoranti* — €X/mese _(attenzione: tendenza in crescita, cerca di contenere)_\n• *Trasporti* — €X/mese _(ottimo: in calo, continua così)_\n\n---\n\n💡 *3 Consigli per questo mese*\n\nBasati ESCLUSIVAMENTE sui trend e anomalie nei dati:\n\n1️⃣ [Consiglio specifico con categoria/importo reale dai dati]\n2️⃣ [Consiglio specifico]\n3️⃣ [Consiglio specifico]\n\n---\n✨ _Generato con AI — verifica sempre i numeri_\n_Scrivi in modo naturale per modificare un obiettivo_\n'
WEEKLY_PROMPT_DATA = "=== PIANO SETTIMANALE: 20/02/2026 → 26/02/2026 ===\nSettimane rimanenti nel mese: 1.3\nTotale speso questo mese: €1228.48\nEntrate totali questo mese: €0.00\nSaldo totale conti: €3240.00\n\nBUDGET MENSILE PER CATEGORIA (limite_reale = residuo / settimane rimaste; soft_cap = limite settimanale ridotto se trend storico in peggioramento, non influenza lo status; status = ok/attenzione/sforato basato su proiezione mensile):\n  Affitto: budget €700.00 | speso mese €700.00 | residuo €0.00 | limite_reale €0.00/s | soft_cap: nessuno | status: attenzione | speso sett. scorsa €0.00 [spesa=0: non mostrare avvisi negativi]\n  Supermercato: budget €220.00 | speso mese €159.00 | residuo €61.00 | limite_reale €47.44/s | soft_cap: nessuno | status: attenzione | speso sett. scorsa €67.80\n  Ristoranti: budget €130.00 | speso mese €185.00 | SFORATO di €55.00 | limite_reale €0/s | status: sforato | speso sett. scorsa €85.00\n  Abbigliamento: budget €100.00 | speso mese €94.00 | residuo €6.00 | limite_reale €4.67/s | soft_cap: nessuno | status: attenzione | speso sett. scorsa €0.00 [spesa=0: non mostrare avvisi negativi]\n  Utility: budget €100.00 | speso mese €0.00 | residuo €100.00 | limite_reale €77.78/s | soft_cap: nessuno | status: ok | speso sett. scorsa €0.00 [spesa=0: non mostrare avvisi negativi]\n  Trasporti: budget €40.00 | speso mese €35.00 | residuo €5.00 | limite_reale €3.89/s | soft_cap: nessuno | status: attenzione | speso sett. scorsa €35.00\n  Abbonamenti: budget €30.00 | speso mese €23.98 | residuo €6.02 | limite_reale €4.68/s | soft_cap: nessuno | status: attenzione | speso sett. scorsa €0.00 [spesa=0: non mostrare avvisi negativi]\n\nPATTERN GIORNALIERI (% spesa per giorno della settimana):\n\nIMPORTANTE: nel report distingui sempre 'limite reale' (derivato dal budget utente) da 'limite consigliato' (riduzione prudenziale per trend in peggioramento). Non presentarli mai come equivalenti."
MONTHLY_PROMPT_DATA = '=== PIANO MENSILE: Marzo 2026 ===\nData riferimento: 20/02/2026\nPrevisione entrate mese: €0.00\nSpese totale mese scorso: €1138.88\nSaldo totale conti: €3240.00\nObiettivo risparmio stimato: €0.00\n\nABBONAMENTI IN SCADENZA NEL MESE (totale €23.98):\n  Netflix: €13.99 — scade 05/03\n  Spotify: €9.99 — scade 05/03\n\nTARGET MENSILE PER CATEGORIA (basato su storico 6 mesi):\n  Affitto: target €700.00/mese | media 6m n/d | trend: stabile | budget mensile €700.00\n  Supermercato: target €220.00/mese | media 6m n/d | trend: stabile | budget mensile €220.00\n  Ristoranti: target €130.00/mese | media 6m n/d | trend: stabile | budget mensile €130.00\n  Abbigliamento: target €100.00/mese | media 6m n/d | trend: stabile | budget mensile €100.00\n  Utility: target €100.00/mese | media 6m n/d | trend: stabile | budget mensile €100.00\n  Trasporti: target €40.00/mese | media 6m n/d | trend: stabile | budget mensile €40.00\n  Abbonamenti: target €30.00/mese | media 6m n/d | trend: stabile | budget mensile €30.00'
WEEK_LABEL = "16/02 → 22/02"
MONTH_LABEL = "Marzo 2026"

def build_user_prompt(message: str) -> str:
    cats_str = ", ".join(CATEGORIES)
    return f'Data di oggi: 2026-04-10\nCategorie disponibili: [{cats_str}]\n\nMessaggio dell\'utente: "{message}"'


## PARTE 1 — gemma2:2b (solo testo: parsing testo + report)

`gemma2:2b` non supporta immagini/audio: partecipa solo ai due task testuali.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GEMMA2_ID = "google/gemma-2-2b-it"
g2_tok = AutoTokenizer.from_pretrained(GEMMA2_ID)
g2_model = AutoModelForCausalLM.from_pretrained(GEMMA2_ID, torch_dtype=torch.bfloat16).to(DEVICE).eval()

def gemma2_chat(system: str | None, user: str, max_new_tokens: int = 1024) -> tuple[str, float]:
    msgs = []
    # gemma2 non supporta il ruolo "system" nel chat template ufficiale: lo prependiamo al messaggio utente
    content = f"{system}\n\n{user}" if system else user
    msgs.append({"role": "user", "content": content})
    inputs = g2_tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(g2_model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = g2_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - t0
    text = g2_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text, elapsed

In [ ]:
# Parsing testo — gemma2:2b
RESULTS["text_parsing"]["gemma2:2b"] = []
for case in TEXT_CASES:
    user_prompt = build_user_prompt(case["msg"])
    text, elapsed = gemma2_chat(SYSTEM_PROMPT, user_prompt, max_new_tokens=512)
    RESULTS["text_parsing"]["gemma2:2b"].append({"msg": case["msg"], "raw_output": text, "elapsed_s": elapsed})
    print(f"[{elapsed:5.1f}s] {case['msg'][:40]:40s} -> {text[:80]}")


In [ ]:
# Report settimanale + mensile — gemma2:2b
weekly_prompt = WEEKLY_TEMPLATE.format(plan_data=WEEKLY_PROMPT_DATA, week_label=WEEK_LABEL)
monthly_prompt = MONTHLY_TEMPLATE.format(plan_data=MONTHLY_PROMPT_DATA, month_label=MONTH_LABEL)

text, elapsed = gemma2_chat(None, weekly_prompt, max_new_tokens=2048)
RESULTS["reports"]["gemma2:2b_weekly"] = {"raw_output": text, "elapsed_s": elapsed}
print(f"weekly [{elapsed:.1f}s]:\n{text[:500]}")

text, elapsed = gemma2_chat(None, monthly_prompt, max_new_tokens=2048)
RESULTS["reports"]["gemma2:2b_monthly"] = {"raw_output": text, "elapsed_s": elapsed}
print(f"monthly [{elapsed:.1f}s]:\n{text[:500]}")


In [ ]:
# Libera VRAM prima di caricare gemma3n
free_model("g2_model", "g2_tok")

## PARTE 2 — gemma3n:e2b (multimodale: testo + report + foto + audio)

`e2b` è la variante "on-device" (footprint ridotto) — coerente con la domanda
originale "gira su un telefono di fascia bassa?". Se vuoi il tetto massimo di qualità,
cambia `GEMMA3N_ID` in `google/gemma-3n-e4b-it` (più pesante, ~8B effettivi) e rilancia
solo questa sezione.

In [ ]:
from transformers import AutoProcessor, Gemma3nForConditionalGeneration

GEMMA3N_ID = "google/gemma-3n-e2b-it"
g3_proc = AutoProcessor.from_pretrained(GEMMA3N_ID)
g3_model = Gemma3nForConditionalGeneration.from_pretrained(GEMMA3N_ID, torch_dtype=torch.bfloat16).to(DEVICE).eval()

def gemma3n_chat(content_blocks: list[dict], system: str | None = None, max_new_tokens: int = 1024,
                  _debug_keys: bool = False) -> tuple[str, float]:
    msgs = []
    if system:
        msgs.append({"role": "system", "content": [{"type": "text", "text": system}]})
    msgs.append({"role": "user", "content": content_blocks})
    inputs = g3_proc.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(g3_model.device)
    if _debug_keys:
        # Sanity check: se manca una chiave tipo "input_features"/"audio_..." qui,
        # il contenuto audio/immagine NON è stato agganciato dal processor — è stato
        # silenziosamente ignorato, e il modello risponde solo al testo (bug capitato
        # con la prima versione di questo notebook: verificare SEMPRE prima di lanciare
        # un intero batch, non dopo).
        print("  [debug] chiavi inputs:", list(inputs.keys()))
    t0 = time.time()
    with torch.inference_mode():
        out = g3_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - t0
    text = g3_proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text, elapsed

In [ ]:
# Parsing testo — gemma3n:e2b
RESULTS["text_parsing"]["gemma3n:e2b"] = []
for case in TEXT_CASES:
    user_prompt = build_user_prompt(case["msg"])
    text, elapsed = gemma3n_chat([{"type": "text", "text": user_prompt}], system=SYSTEM_PROMPT, max_new_tokens=512)
    RESULTS["text_parsing"]["gemma3n:e2b"].append({"msg": case["msg"], "raw_output": text, "elapsed_s": elapsed})
    print(f"[{elapsed:5.1f}s] {case['msg'][:40]:40s} -> {text[:80]}")


In [ ]:
# Report settimanale + mensile — gemma3n:e2b
text, elapsed = gemma3n_chat([{"type": "text", "text": weekly_prompt}], max_new_tokens=2048)
RESULTS["reports"]["gemma3n:e2b_weekly"] = {"raw_output": text, "elapsed_s": elapsed}
print(f"weekly [{elapsed:.1f}s]:\n{text[:500]}")

text, elapsed = gemma3n_chat([{"type": "text", "text": monthly_prompt}], max_new_tokens=2048)
RESULTS["reports"]["gemma3n:e2b_monthly"] = {"raw_output": text, "elapsed_s": elapsed}
print(f"monthly [{elapsed:.1f}s]:\n{text[:500]}")


### Foto — genera scontrini sintetici e chiedi a gemma3n il JSON transazione completo

Un solo hop: immagine -> JSON strutturato (stesso schema di `expense_parser.py`),
niente OCR intermedio — è il modo naturale di usare un modello multimodale, ed è la
differenza che vogliamo misurare rispetto alla pipeline attuale (Tesseract + Mistral).

In [ ]:
from PIL import Image, ImageDraw, ImageFont

# Font di sistema a dimensione leggibile — ImageFont.load_default() genera un
# bitmap font minuscolo (~8px), illeggibile anche per un umano allo zoom nativo:
# nel primo giro di test gemma3n non ha MAI letto correttamente la data dello
# scontrino, sempre sostituita con la data odierna del prompt — probabile causa,
# non necessariamente un limite del modello. Font TrueType a 28px + upscale 2x
# per dare al vision encoder margine sufficiente a distinguere i caratteri.
def _load_font(size: int) -> ImageFont.FreeTypeFont:
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeMono.ttf",
    ]:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    raise RuntimeError("Nessun font TrueType di sistema trovato — impossibile generare scontrini leggibili")

def render_receipt(case: dict) -> Image.Image:
    font = _load_font(28)
    font_bold = _load_font(30)
    lines = [(case["store"].upper(), True), ("-" * 32, False), ("SCONTRINO FISCALE", True), ("-" * 32, False)]
    for name, price in case["items"]:
        lines.append((f"{name:<22}{price:>8}", False))
    lines.append(("-" * 32, False))
    lines.append((f"TOTALE EUR{case['total']:>20}", True))
    lines.append(("-" * 32, False))
    lines.append((case["date"], False))
    line_height = 38
    img = Image.new("RGB", (720, len(lines) * line_height + 40), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    y = 20
    for text_line, bold in lines:
        draw.text((20, y), text_line, fill=(10, 10, 10), font=font_bold if bold else font)
        y += line_height
    return img

PHOTO_PROMPT = (
    SYSTEM_PROMPT
    + '\n\nEstrai la transazione da questa foto di scontrino. '
    + f'Data di oggi: 2026-04-10\nCategorie disponibili: [{", ".join(CATEGORIES)}]'
)

RESULTS["photo"]["gemma3n:e2b"] = []
for i, case in enumerate(PHOTO_CASES):
    img = render_receipt(case)
    if i == 0:
        display(img)  # controllo visivo: verifica TU STESSO che sia leggibile prima di lanciare tutto il batch
    text, elapsed = gemma3n_chat(
        [{"type": "image", "image": img}, {"type": "text", "text": PHOTO_PROMPT}],
        max_new_tokens=512, _debug_keys=(i == 0),
    )
    RESULTS["photo"]["gemma3n:e2b"].append({
        "ground_truth": case, "raw_output": text, "elapsed_s": elapsed,
    })
    print(f"[{elapsed:5.1f}s] {case['store']:25s} -> {text[:100]}")


### Audio — genera frasi vocali sintetiche (gTTS) e chiedi a gemma3n il JSON transazione completo

In [ ]:
!pip install -q soundfile librosa

from gtts import gTTS
import librosa
import soundfile as sf

AUDIO_PROMPT = (
    SYSTEM_PROMPT
    + '\n\nTrascrivi ed estrai la transazione da questo audio. '
    + f'Data di oggi: 2026-04-10\nCategorie disponibili: [{", ".join(CATEGORIES)}]'
)

# IMPORTANTE: passiamo un PERCORSO FILE ("path"), non un array numpy pre-caricato.
# Nel primo giro, un array grezzo passato sotto {"type": "audio", "audio": wav} è
# stato silenziosamente ignorato dal processor (5 risposte su 6 identiche, il
# modello rispondeva solo al testo) — passare il path lascia che sia il processor
# stesso a caricare/ricampionare l'audio nel formato che si aspetta internamente,
# stesso pattern documentato e funzionante per i video multimodali in transformers.
RESULTS["audio"]["gemma3n:e2b"] = []
for i, case in enumerate(AUDIO_CASES):
    mp3_path = f"/content/audio_{i}.mp3"
    wav_path = f"/content/audio_{i}.wav"
    gTTS(text=case["text"], lang="it").save(mp3_path)
    wav, sr = librosa.load(mp3_path, sr=16000)  # solo per convertire mp3 -> wav pulito
    sf.write(wav_path, wav, sr)

    text, elapsed = gemma3n_chat(
        [{"type": "audio", "path": wav_path}, {"type": "text", "text": AUDIO_PROMPT}],
        max_new_tokens=512, _debug_keys=(i == 0),
    )
    RESULTS["audio"]["gemma3n:e2b"].append({
        "ground_truth": case, "raw_output": text, "elapsed_s": elapsed,
    })
    print(f"[{elapsed:5.1f}s] {case['text'][:40]:40s} -> {text[:100]}")


## Salva ed esporta i risultati

Scarica `gemma_colab_results.json` e dallo a Claude in sessione locale: verrà
punteggiato con le stesse funzioni di scoring usate per Mistral (nessun doppio
standard tra provider).

In [ ]:
with open("/content/gemma_colab_results.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download("/content/gemma_colab_results.json")
print("Fatto. Carica gemma_colab_results.json nella conversazione con Claude.")